In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import warnings
warnings.filterwarnings('ignore') 

print("Library berhasil diimport!")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Load dataset
def load_split_data(split_num):
    """Load data untuk split tertentu""" 
    X_train = pd.read_csv(f'feature-engineering/X0-train-split{split_num}-seqglo-truncate.csv')
    X_test = pd.read_csv(f'feature-engineering/X0-test-split{split_num}-seqglo-truncate.csv')
    y_train = pd.read_csv(f'feature-engineering/y0-train-split{split_num}-seqglo-truncate.csv')
    y_test = pd.read_csv(f'feature-engineering/y0-test-split{split_num}-seqglo-truncate.csv')
    
    return X_train, X_test, y_train, y_test

# Load split pertama sebagai contoh
X_train, X_test, y_train, y_test = load_split_data(1)

print(f"Shape X_train: {X_train.shape}")
print(f"Shape X_test: {X_test.shape}")
print(f"Shape y_train: {y_train.shape}")
print(f"Shape y_test: {y_test.shape}")
print(f"\nFeatures: {list(X_train.columns)}")
print(f"Unique labels: {sorted(y_train['label'].unique())}")

In [ ]:
# Fungsi untuk membuat window biasa (non-overlapping)
def create_non_overlapping_window(data, window_size=60):
    """
    Membuat window biasa (non-overlapping) untuk data time series
    
    Args:
        data: numpy array atau pandas DataFrame
        window_size: ukuran window (default 60)
    
    Returns:
        X: array 3D dengan shape (samples, window_size, features)
        y: array 1D dengan label untuk setiap window
    """
    if isinstance(data, pd.DataFrame):
        features = data.drop('label', axis=1).values if 'label' in data.columns else data.values
        labels = data['label'].values if 'label' in data.columns else None
    else:
        features = data
        labels = None
    
    X, y = [], []
    
    # Hitung jumlah window yang bisa dibuat
    num_windows = len(features) // window_size
    
    for i in range(num_windows):
        start_idx = i * window_size
        end_idx = start_idx + window_size
        
        # Ambil window
        window_features = features[start_idx:end_idx]
        X.append(window_features)
        
        if labels is not None:
            # Ambil label terakhir dari window atau mayoritas label dalam window
            window_labels = labels[start_idx:end_idx]
            # Gunakan label terakhir dari window
            y.append(window_labels[-1])
            
            # Alternatif: gunakan mayoritas voting
            # from scipy import stats
            # y.append(stats.mode(window_labels)[0][0])
    
    return np.array(X), np.array(y)

# Test fungsi window biasa
print("Testing non-overlapping window function...")
sample_data = pd.concat([X_train.head(120), y_train.head(120)], axis=1)  # Ambil 120 sample untuk 2 window
X_sample_non_overlap, y_sample_non_overlap = create_non_overlapping_window(sample_data, window_size=60)
print(f"Original data shape: {sample_data.shape}")
print(f"Non-overlapping windowed X shape: {X_sample_non_overlap.shape}")
print(f"Non-overlapping windowed y shape: {y_sample_non_overlap.shape}")
print(f"Reduction factor: {sample_data.shape[0] / X_sample_non_overlap.shape[0]:.1f}x")

In [ ]:
# Data preprocessing dengan window biasa (non-overlapping)
def preprocess_data_non_overlapping(X_train, X_test, y_train, y_test, window_size=60):
    """
    Preprocessing data untuk LSTM dengan window biasa (non-overlapping)
    """
    print("Melakukan preprocessing data dengan non-overlapping windows...")
    
    # Copy data untuk menghindari modifikasi original
    X_train_copy = X_train.copy()
    X_test_copy = X_test.copy()
    y_train_copy = y_train.copy()
    y_test_copy = y_test.copy()
    
    # Cek dan handle missing values
    print(f"Missing values in X_train: {X_train_copy.isnull().sum().sum()}")
    print(f"Missing values in X_test: {X_test_copy.isnull().sum().sum()}")
    
    # Fill missing values dengan median
    if X_train_copy.isnull().sum().sum() > 0:
        X_train_copy = X_train_copy.fillna(X_train_copy.median())
    if X_test_copy.isnull().sum().sum() > 0:
        X_test_copy = X_test_copy.fillna(X_train_copy.median())
    
    # Normalisasi fitur - HANYA fit pada training data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_copy)
    X_test_scaled = scaler.transform(X_test_copy)
    
    print(f"Training data - Mean: {X_train_scaled.mean():.6f}, Std: {X_train_scaled.std():.6f}")
    print(f"Testing data - Mean: {X_test_scaled.mean():.6f}, Std: {X_test_scaled.std():.6f}")
    
    # Convert back to DataFrame
    X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_copy.columns)
    X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test_copy.columns)
    
    # Gabungkan dengan labels
    train_data = pd.concat([X_train_scaled_df, y_train_copy.reset_index(drop=True)], axis=1)
    test_data = pd.concat([X_test_scaled_df, y_test_copy.reset_index(drop=True)], axis=1)
    
    # Buat non-overlapping windows
    print(f"Membuat non-overlapping windows dengan ukuran {window_size}...")
    X_train_window, y_train_window = create_non_overlapping_window(train_data, window_size)
    X_test_window, y_test_window = create_non_overlapping_window(test_data, window_size)
    
    print(f"Data reduction:")
    print(f"  Training: {len(train_data)} -> {len(X_train_window)} windows ({len(train_data)/len(X_train_window):.1f}x reduction)")
    print(f"  Testing: {len(test_data)} -> {len(X_test_window)} windows ({len(test_data)/len(X_test_window):.1f}x reduction)")
    
    # Encode labels
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train_window)
    y_test_encoded = label_encoder.transform(y_test_window)
    
    print(f"Window shapes - X_train: {X_train_window.shape}, X_test: {X_test_window.shape}")
    print(f"Label distribution - Train: {np.bincount(y_train_encoded)}, Test: {np.bincount(y_test_encoded)}")
    
    return X_train_window, X_test_window, y_train_encoded, y_test_encoded, scaler, label_encoder

# Preprocessing data dengan non-overlapping windows
print("=== Preprocessing data dengan Non-Overlapping Windows ===")
X_train_proc_non_overlap, X_test_proc_non_overlap, y_train_proc_non_overlap, y_test_proc_non_overlap, scaler_non_overlap, label_encoder_non_overlap = preprocess_data_non_overlapping(
    X_train, X_test, y_train, y_test, window_size=60
)

print(f"\nHasil preprocessing dengan non-overlapping windows:")
print(f"X_train_processed shape: {X_train_proc_non_overlap.shape}")
print(f"X_test_processed shape: {X_test_proc_non_overlap.shape}")
print(f"y_train_processed shape: {y_train_proc_non_overlap.shape}")
print(f"y_test_processed shape: {y_test_proc_non_overlap.shape}")
print(f"Number of classes: {len(label_encoder_non_overlap.classes_)}")
print(f"Classes: {label_encoder_non_overlap.classes_}")

In [ ]:
# Model LSTM untuk non-overlapping windows
def create_lstm_for_non_overlapping(input_shape, num_classes):
    """
    Membuat model LSTM yang dioptimalkan untuk non-overlapping windows
    Karena data lebih sedikit, model bisa sedikit lebih kompleks
    """
    model = Sequential([
        # Layer LSTM pertama - sedikit lebih besar karena data berkurang
        LSTM(96, return_sequences=True, input_shape=input_shape,
             dropout=0.2, recurrent_dropout=0.2),
        BatchNormalization(),
        
        # Layer LSTM kedua  
        LSTM(48, return_sequences=False,
             dropout=0.3, recurrent_dropout=0.3),
        BatchNormalization(),
        
        # Dense layers
        Dense(48, activation='relu'),
        Dropout(0.4),
        BatchNormalization(),
        
        Dense(24, activation='relu'),
        Dropout(0.3),
        
        # Output layer
        Dense(num_classes, activation='softmax')
    ])
    
    return model

# Buat model untuk non-overlapping windows
input_shape_non_overlap = (X_train_proc_non_overlap.shape[1], X_train_proc_non_overlap.shape[2])
num_classes_non_overlap = len(label_encoder_non_overlap.classes_)

model_non_overlap = create_lstm_for_non_overlapping(input_shape_non_overlap, num_classes_non_overlap)

# Compile model
model_non_overlap.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model Summary (Non-Overlapping Windows):")
model_non_overlap.summary()
print(f"\nInput shape: {input_shape_non_overlap}")
print(f"Number of classes: {num_classes_non_overlap}")
print(f"Total parameters: {model_non_overlap.count_params():,}")

In [ ]:
 # Training dengan non-overlapping windows dan ModelCheckpoint
def train_non_overlapping_model(model, X_train, y_train, X_test, y_test, checkpoint_path='model_checkpoints/non_overlap_best_model.h5'):
    """
    Training model dengan non-overlapping windows
    """
    import os
    
    # Buat direktori untuk checkpoint jika belum ada
    checkpoint_dir = os.path.dirname(checkpoint_path)
    if checkpoint_dir and not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)
    
    # Callbacks
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1,
        mode='min'
    )
    
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1,
        mode='min'
    )
    
    model_checkpoint = ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=False,
        mode='min',
        verbose=1
    )
    
    print(f"=== Training Non-Overlapping Window Model ===")
    print(f"Training samples: {X_train.shape[0]:,}")
    print(f"Validation samples: {X_test.shape[0]:,}")
    print(f"Model checkpoint: {checkpoint_path}")
    
    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,  # Batch size lebih kecil karena data berkurang
        validation_data=(X_test, y_test),
        callbacks=[early_stopping, reduce_lr, model_checkpoint],
        verbose=1,
        shuffle=True
    )
    
    return history, checkpoint_path

# Training model non-overlapping
print("=== Memulai Training Model Non-Overlapping Windows ===")
history_non_overlap, saved_checkpoint_non_overlap = train_non_overlapping_model(
    model_non_overlap,
    X_train_proc_non_overlap, y_train_proc_non_overlap,
    X_test_proc_non_overlap, y_test_proc_non_overlap
)

print(f"\nTraining selesai!")
print(f"Model terbaik disimpan di: {saved_checkpoint_non_overlap}")
print(f"Final training loss: {history_non_overlap.history['loss'][-1]:.4f}")
print(f"Final validation loss: {history_non_overlap.history['val_loss'][-1]:.4f}")
print(f"Final training accuracy: {history_non_overlap.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history_non_overlap.history['val_accuracy'][-1]:.4f}")

In [ ]:
# Evaluasi model yang diperbaiki
print("=== Evaluasi Model yang Diperbaiki ===")
accuracy_improved, y_pred_improved, y_pred_proba_improved = evaluate_model(
    model_improved, X_test_proc_new, y_test_proc_new, label_encoder_new
)

print(f"\n=== Perbandingan Hasil ===")
print(f"Model Lama - Test Accuracy: {accuracy:.4f}")
print(f"Model Baru - Test Accuracy: {accuracy_improved:.4f}")
print(f"Improvement: {accuracy_improved - accuracy:.4f}")

# Cek overfitting dengan melihat gap antara training dan validation
final_train_acc = history_improved.history['accuracy'][-1]
final_val_acc = history_improved.history['val_accuracy'][-1]
overfitting_gap = final_train_acc - final_val_acc

print(f"\n=== Analisis Overfitting ===")
print(f"Final Training Accuracy: {final_train_acc:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")
print(f"Overfitting Gap: {overfitting_gap:.4f}")

if overfitting_gap < 0.1:
    print("✅ Overfitting terkendali (gap < 10%)")
elif overfitting_gap < 0.2:
    print("⚠️ Overfitting sedang (gap 10-20%)")
else:
    print("❌ Overfitting tinggi (gap > 20%)")

In [ ]:
# Visualisasi perbandingan training history
def plot_comparison_history(history_old, history_new):
    """
    Plot perbandingan training history antara model lama dan baru
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss comparison
    ax1.plot(history_old.history['loss'], label='Old Model - Training Loss', alpha=0.7)
    ax1.plot(history_old.history['val_loss'], label='Old Model - Validation Loss', alpha=0.7)
    ax1.set_title('Model Lama - Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(history_new.history['loss'], label='New Model - Training Loss', alpha=0.7)
    ax2.plot(history_new.history['val_loss'], label='New Model - Validation Loss', alpha=0.7)
    ax2.set_title('Model Baru - Loss (Improved)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)
    
    # Accuracy comparison
    ax3.plot(history_old.history['accuracy'], label='Old Model - Training Acc', alpha=0.7)
    ax3.plot(history_old.history['val_accuracy'], label='Old Model - Validation Acc', alpha=0.7)
    ax3.set_title('Model Lama - Accuracy')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Accuracy')
    ax3.legend()
    ax3.grid(True)
    
    ax4.plot(history_new.history['accuracy'], label='New Model - Training Acc', alpha=0.7)
    ax4.plot(history_new.history['val_accuracy'], label='New Model - Validation Acc', alpha=0.7)
    ax4.set_title('Model Baru - Accuracy (Improved)')
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Accuracy')
    ax4.legend()
    ax4.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Plot overfitting analysis
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Calculate overfitting gap over epochs
    old_gap = np.array(history_old.history['accuracy']) - np.array(history_old.history['val_accuracy'])
    new_gap = np.array(history_new.history['accuracy']) - np.array(history_new.history['val_accuracy'])
    
    ax1.plot(old_gap, label='Model Lama', alpha=0.7, color='red')
    ax1.plot(new_gap, label='Model Baru', alpha=0.7, color='green')
    ax1.set_title('Overfitting Gap (Training Acc - Validation Acc)')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Gap')
    ax1.legend()
    ax1.grid(True)
    ax1.axhline(y=0.1, color='orange', linestyle='--', alpha=0.5, label='10% threshold')
    
    # Final comparison bar chart
    categories = ['Final Train Acc', 'Final Val Acc', 'Test Acc']
    old_scores = [history_old.history['accuracy'][-1], 
                  history_old.history['val_accuracy'][-1], 
                  accuracy]
    new_scores = [history_new.history['accuracy'][-1], 
                  history_new.history['val_accuracy'][-1], 
                  accuracy_improved]
    
    x = np.arange(len(categories))
    width = 0.35
    
    ax2.bar(x - width/2, old_scores, width, label='Model Lama', alpha=0.7, color='red')
    ax2.bar(x + width/2, new_scores, width, label='Model Baru', alpha=0.7, color='green')
    ax2.set_title('Perbandingan Performa Final')
    ax2.set_ylabel('Accuracy')
    ax2.set_xticks(x)
    ax2.set_xticklabels(categories)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for i, (old, new) in enumerate(zip(old_scores, new_scores)):
        ax2.text(i - width/2, old + 0.01, f'{old:.3f}', ha='center', va='bottom')
        ax2.text(i + width/2, new + 0.01, f'{new:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

# Plot perbandingan jika sudah ada history lama
if 'history' in locals():
    print("=== Visualisasi Perbandingan Model ===")
    plot_comparison_history(history, history_improved)
else:
    print("=== Visualisasi Model Baru ===")
    plot_training_history(history_improved)

In [ ]:
# Evaluasi model
def evaluate_model(model, X_test, y_test, label_encoder):
    """
    Evaluasi model dan tampilkan metrik
    """
    # Prediksi
    y_pred_proba = model.predict(X_test)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Test Accuracy: {accuracy:.4f}")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, 
                              target_names=label_encoder.classes_.astype(str)))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    
    return accuracy, y_pred, y_pred_proba

# Evaluasi model
accuracy, y_pred, y_pred_proba = evaluate_model(
    model, X_test_processed, y_test_processed, label_encoder
)

In [ ]:
# Visualisasi training history
def plot_training_history(history):
    """
    Plot training dan validation loss/accuracy
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss plot
    ax1.plot(history.history['loss'], label='Training Loss')
    ax1.plot(history.history['val_loss'], label='Validation Loss')
    ax1.set_title('Model Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    
    # Accuracy plot
    ax2.plot(history.history['accuracy'], label='Training Accuracy')
    ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax2.set_title('Model Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

# Plot training history
plot_training_history(history)

In [ ]:
# Cross-validation dengan multiple splits
def run_cross_validation(num_splits=5, window_size=60):
    """
    Menjalankan cross-validation dengan semua split data
    """
    results = []
    
    for split in range(1, num_splits + 1):
        print(f"\n=== Training Split {split} ===")
        
        # Load data untuk split ini
        X_train_cv, X_test_cv, y_train_cv, y_test_cv = load_split_data(split)
        
        # Preprocessing
        X_train_proc, X_test_proc, y_train_proc, y_test_proc, scaler_cv, le_cv = preprocess_data(
            X_train_cv, X_test_cv, y_train_cv, y_test_cv, window_size
        )
        
        # Buat model baru untuk setiap split
        input_shape = (X_train_proc.shape[1], X_train_proc.shape[2])
        num_classes = len(le_cv.classes_)
        
        model_cv = create_lstm_model(input_shape, num_classes)
        model_cv.compile(
            optimizer=Adam(learning_rate=0.001),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # Training
        history_cv = train_model(
            model_cv, X_train_proc, y_train_proc, 
            X_test_proc, y_test_proc,
            epochs=50,  # Kurangi epoch untuk CV
            batch_size=32
        )
        
        # Evaluasi
        y_pred_proba = model_cv.predict(X_test_proc)
        y_pred = np.argmax(y_pred_proba, axis=1)
        accuracy = accuracy_score(y_test_proc, y_pred)
        
        results.append({
            'split': split,
            'accuracy': accuracy,
            'model': model_cv,
            'history': history_cv,
            'scaler': scaler_cv,
            'label_encoder': le_cv
        })
        
        print(f"Split {split} Accuracy: {accuracy:.4f}")
    
    # Ringkasan hasil
    accuracies = [r['accuracy'] for r in results]
    print(f"\n=== Cross-Validation Results ===")
    print(f"Mean Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
    print(f"Best Split: {np.argmax(accuracies) + 1} (Accuracy: {np.max(accuracies):.4f})")
    
    return results

# Jalankan cross-validation (opsional - uncomment jika ingin menjalankan)
# cv_results = run_cross_validation(num_splits=5, window_size=60)